# 08 · Normalize billing, settlement and general ledger

Run this notebook from the `silver/` folder with the existing project kernel. It selects and verifies Bronze manifests, then rebuilds the required prior stages in memory. Reuse the same `silver_settings.json` to review a consistent selection across notebooks.

In [1]:
from IPython.display import display
import pandas as pd

from silver_notebook import build, frame, input_frame, settings_from_file

model = build("finance", settings_from_file())
display(input_frame(model))

,source,manifest
0,erpnext,batch/erpnext/manual__2026-11-21T15%3A27%3A44....
1,dealer-sql,batch/dealer-sql/manual__2026-11-21T15%3A28%3A...
2,parent-pos,batch/parent-pos/manual__2026-11-21T15%3A27%3A...
3,dealer_export,files/otc-inbound/dealer/exports/dealer-export...
4,parent_finance,files/otc-inbound/parent/finance/dealer-order-...
5,dealer_finance,files/otc-inbound/dealer/finance/dealer-order-...
6,mdm,files/otc-inbound/seed-v1/8fab9f89f5242c6819f2...


Billing documents include invoices and credits. ERPNext supplies billing lines and original invoice references where recorded. Dealer billing is document-level. A dealer credit with no original invoice ID remains unassigned at invoice grain.

Payment Allocation is the bridge between a payment and an invoice. GL posts at posting-row grain. Reconstruct AR by posting cutoff in notebook 10 and compare with a dealer observation for the same date. ERPNext current outstanding comes from the extraction and is retained as a separate field.

In [2]:
display(frame(model, "billing_document"), frame(model, "billing_document_line"))
display(frame(model, "payment"), frame(model, "payment_allocation"))
display(frame(model, "gl_posting_line"), frame(model, "receivables_snapshot"))
display(pd.DataFrame(model.exceptions))

,id,legal_entity_id,source_id,source_reference,document_type,sales_order_id,source_customer_id,posting_date,due_date,currency,docstatus,signed_total,source_current_outstanding,original_invoice_source_id,original_invoice_id,line_coverage
0,"[""parent"",""ACC-SINV-2026-00001""]",parent,ACC-SINV-2026-00001,P-I1,invoice,"[""parent"",""SAL-ORD-2026-00001""]","[""parent"",""OTC P-C1 Metro Industrial Services""]",2026-11-03,2026-12-03,INR,1,60000.0,35000.0,None,None,source lines
1,"[""parent"",""ACC-SINV-2026-00002""]",parent,ACC-SINV-2026-00002,P-I2,invoice,"[""parent"",""SAL-ORD-2026-00001""]","[""parent"",""OTC P-C1 Metro Industrial Services""]",2026-11-05,2026-12-05,INR,1,40000.0,35000.0,None,None,source lines
2,"[""parent"",""ACC-SINV-2026-00003""]",parent,ACC-SINV-2026-00003,P-CR1,credit_note,None,"[""parent"",""OTC P-C1 Metro Industrial Services""]",2026-11-15,2026-11-15,INR,1,-5000.0,0.0,ACC-SINV-2026-00002,"[""parent"",""ACC-SINV-2026-00002""]",source lines
3,"[""parent"",""ACC-SINV-2026-00004""]",parent,ACC-SINV-2026-00004,P-BRONZE-R2-INVOICE,invoice,"[""parent"",""SAL-ORD-2026-00006""]","[""parent"",""OTC P-C1 Metro Industrial Services""]",2026-11-15,2026-11-15,INR,1,20000.0,10000.0,None,None,source lines
4,"[""parent"",""ACC-SINV-2026-00005""]",parent,ACC-SINV-2026-00005,P-BRONZE-R2-CREDIT,credit_note,None,"[""parent"",""OTC P-C1 Metro Industrial Services""]",2026-11-15,2026-11-15,INR,1,-10000.0,0.0,ACC-SINV-2026-00004,"[""parent"",""ACC-SINV-2026-00004""]",source lines
5,"[""dealer"",""D-BRONZE-R2-CREDIT""]",dealer,D-BRONZE-R2-CREDIT,D-BRONZE-R2-CREDIT,credit_note,"[""dealer"",""D-BRONZE-R2-ORDER""]","[""dealer"",""D-C1""]",2026-11-15,2026-12-15,INR,1,-10000,None,None,None,document only
6,"[""dealer"",""D-BRONZE-R2-INVOICE""]",dealer,D-BRONZE-R2-INVOICE,D-BRONZE-R2-INVOICE,invoice,"[""dealer"",""D-BRONZE-R2-ORDER""]","[""dealer"",""D-C1""]",2026-11-15,2026-12-15,INR,1,20000,None,None,None,document only
7,"[""dealer"",""D-I1""]",dealer,D-I1,D-I1,invoice,"[""dealer"",""D-S1""]","[""dealer"",""D-C1""]",2026-11-05,2026-12-05,INR,1,40000,None,None,None,document only


,id,billing_document_id,sales_order_line_id,shipment_line_id,source_product_id,original_billing_line_id,quantity,uom,amount
0,"[""parent"",""ACC-SINV-2026-00001"",""vqqiva6run""]","[""parent"",""ACC-SINV-2026-00001""]","[""parent"",""SAL-ORD-2026-00001"",""vqgdj6mnv1""]","[""parent"",""MAT-DN-2026-00001"",""vqnei0r1v0""]","[""parent"",""P-PUMP""]",None,6.0,Nos,60000.0
1,"[""parent"",""ACC-SINV-2026-00002"",""vqv1hei16v""]","[""parent"",""ACC-SINV-2026-00002""]","[""parent"",""SAL-ORD-2026-00001"",""vqgdj6mnv1""]","[""parent"",""MAT-DN-2026-00002"",""vqtf30brbp""]","[""parent"",""P-PUMP""]",None,4.0,Nos,40000.0
2,"[""parent"",""ACC-SINV-2026-00003"",""vr3dokqbr3""]","[""parent"",""ACC-SINV-2026-00003""]",None,None,"[""parent"",""P-PUMP""]","[""parent"",""ACC-SINV-2026-00002"",""vqv1hei16v""]",-1.0,Nos,-5000.0
3,"[""parent"",""ACC-SINV-2026-00004"",""brsj8et66c""]","[""parent"",""ACC-SINV-2026-00004""]","[""parent"",""SAL-ORD-2026-00006"",""brbeshsujo""]","[""parent"",""MAT-DN-2026-00003"",""brlofkakoo""]","[""parent"",""P-PUMP""]",None,2.0,Nos,20000.0
4,"[""parent"",""ACC-SINV-2026-00005"",""bs1dfo2t3s""]","[""parent"",""ACC-SINV-2026-00005""]",None,None,"[""parent"",""P-PUMP""]","[""parent"",""ACC-SINV-2026-00004"",""brsj8et66c""]",-1.0,Nos,-10000.0


,id,legal_entity_id,source_id,payment_type,source_customer_id,payment_reference,posting_date,currency,amount,docstatus
0,"[""parent"",""ACC-PAY-2026-00001""]",parent,ACC-PAY-2026-00001,Receive,"[""parent"",""OTC P-C1 Metro Industrial Services""]",P-P1,2026-11-10,INR,25000.0,1
1,"[""dealer"",""D-R1""]",dealer,D-R1,Receive,None,D-P1,2026-11-12,INR,10000,1


,id,payment_id,billing_document_id,allocated_amount
0,"[""parent"",""ACC-PAY-2026-00001"",""vr1ekg0uio""]","[""parent"",""ACC-PAY-2026-00001""]","[""parent"",""ACC-SINV-2026-00001""]",25000.0
1,"[""dealer"",""D-A1""]","[""dealer"",""D-R1""]","[""dealer"",""D-I1""]",10000


,id,legal_entity_id,source_id,voucher_source_id,voucher_type,posting_date,is_cancelled,company_currency,gl_account_id,billing_document_id,payment_id,debit,credit
0,"[""parent"",""09029f86af""]",parent,09029f86af,MAT-STE-2026-00003,Stock Entry,2026-11-15,False,INR,"[""parent"",""Stock In Hand - OPD""]",None,None,12000.0,0.0
1,"[""parent"",""3b7be7fe71""]",parent,3b7be7fe71,MAT-DN-2026-00003,Delivery Note,2026-11-15,False,INR,"[""parent"",""Stock In Hand - OPD""]",None,None,0.0,12000.0
2,"[""parent"",""6eb02aaaa3""]",parent,6eb02aaaa3,ACC-SINV-2026-00004,Sales Invoice,2026-11-15,False,INR,"[""parent"",""Sales - OPD""]","[""parent"",""ACC-SINV-2026-00004""]",None,0.0,20000.0
3,"[""parent"",""7acfb1ad2a""]",parent,7acfb1ad2a,MAT-DN-2026-00003,Delivery Note,2026-11-15,False,INR,"[""parent"",""Cost of Goods Sold - OPD""]",None,None,12000.0,0.0
4,"[""parent"",""93614da8ce""]",parent,93614da8ce,ACC-SINV-2026-00005,Sales Invoice,2026-11-15,False,INR,"[""parent"",""Sales - OPD""]","[""parent"",""ACC-SINV-2026-00005""]",None,10000.0,0.0
5,"[""parent"",""981f76fdd4""]",parent,981f76fdd4,ACC-SINV-2026-00004,Sales Invoice,2026-11-15,False,INR,"[""parent"",""Debtors - OPD""]","[""parent"",""ACC-SINV-2026-00004""]",None,20000.0,0.0
6,"[""parent"",""ACC-GLE-2026-00001""]",parent,ACC-GLE-2026-00001,MAT-STE-2026-00001,Stock Entry,2026-11-01,False,INR,"[""parent"",""Stock In Hand - OPD""]",None,None,36000.0,0.0
7,"[""parent"",""ACC-GLE-2026-00002""]",parent,ACC-GLE-2026-00002,MAT-STE-2026-00001,Stock Entry,2026-11-01,False,INR,"[""parent"",""Cost of Goods Sold - OPD""]",None,None,0.0,36000.0
8,"[""parent"",""ACC-GLE-2026-00003""]",parent,ACC-GLE-2026-00003,MAT-STE-2026-00002,Stock Entry,2026-11-01,False,INR,"[""parent"",""Stock In Hand - OPD""]",None,None,24000.0,0.0
9,"[""parent"",""ACC-GLE-2026-00004""]",parent,ACC-GLE-2026-00004,MAT-STE-2026-00002,Stock Entry,2026-11-01,False,INR,"[""parent"",""Cost of Goods Sold - OPD""]",None,None,0.0,24000.0


,id,legal_entity_id,cutoff,balance_source,billing_document_id,open_amount,currency
0,"[""dealer"",""D-I1"",""2026-12-10"",""dealer export""]",dealer,2026-12-10,dealer export,"[""dealer"",""D-I1""]",30000,INR
1,"[""dealer"",""D-BRONZE-R2-INVOICE"",""2026-12-16"",""...",dealer,2026-12-16,dealer export,"[""dealer"",""D-BRONZE-R2-INVOICE""]",10000,INR


,rule,table,id,severity,detail
0,unresolved_reference,billing_document,"[""dealer"",""D-BRONZE-R2-CREDIT""]",review,"{'role': 'original_invoice', 'target': 'billin..."
